# Customer Churn Analysis - Phase 4: Modelling

---

## What this notebook does

We take the feature matrix produced in Phase 3 and build a complete, production-grade churn prediction pipeline. Every model choice, metric decision, and tuning strategy is explained with its reasoning.

## The pipeline

```
1.  Load X_raw, X_scaled, y, feature_names from processed/
2.  Train / validation / test split  (64% / 16% / 20%)
3.  Define evaluation metrics         (why ROC-AUC + PR-AUC, not accuracy)
4.  Baseline model                    (floor every model must beat)
5.  SMOTE oversampling preparation    (training set only)
6.  5-fold CV comparison of 4 models
7.  SMOTE vs class_weight comparison  (best model only)
8.  Hyperparameter tuning             (RandomizedSearchCV)
9.  Decision threshold optimisation   (validation set only)
10. Final test set evaluation         (test set touched exactly once)
11. Model interpretation              (SHAP values)
12. At-risk customer scoring          (business deliverable)
13. Save all artefacts
```

## Why the test set is touched exactly once

The test set is a one-shot resource. Every decision you make after seeing test results is leakage. Everything in steps 4-9 happens entirely on training and validation data. The test set is opened only in step 10 to report final numbers. Those are the numbers you put in your portfolio.

## What we inherit from Phase 3

| File | Contents |
|---|---|
| `X_raw.csv` | Unscaled features for tree-based models |
| `X_scaled.csv` | StandardScaler features for linear/distance models |
| `y.csv` | Binary churn target (0/1), ~26% positive rate |
| `scaler.pkl` | Fitted scaler - refit on train-only here |
| `feature_names.txt` | Ordered list of feature column names |

---
## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install packages not included in Colab by default
import subprocess, sys

def _install(pkg):
    subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"], check=True)

try:
    import xgboost
except ImportError:
    _install("xgboost")

try:
    import lightgbm
except ImportError:
    _install("lightgbm")

try:
    import imblearn
except ImportError:
    _install("imbalanced-learn")

try:
    import shap
except ImportError:
    _install("shap")

print("All packages available.")

In [ ]:
import os, warnings, time, joblib
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

import xgboost  as xgb
import lightgbm as lgb
import shap

from imblearn.over_sampling import SMOTE

from sklearn.dummy           import DummyClassifier
from sklearn.linear_model    import LogisticRegression
from sklearn.ensemble        import RandomForestClassifier
from sklearn.preprocessing   import StandardScaler
from sklearn.model_selection  import (
    train_test_split, StratifiedKFold,
    cross_validate, RandomizedSearchCV
)
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    f1_score, recall_score, precision_score,
    classification_report, confusion_matrix,
    RocCurveDisplay, PrecisionRecallDisplay,
    ConfusionMatrixDisplay, precision_recall_curve
)

warnings.filterwarnings("ignore")

BASE_PATH      = "/content/drive/MyDrive/customer-churn-project"
PROCESSED_PATH = os.path.join(BASE_PATH, "data/processed/")
MODELS_PATH    = os.path.join(BASE_PATH, "models/")
os.makedirs(MODELS_PATH, exist_ok=True)

# Plot style consistent with notebooks 02 and 03
sns.set_theme(style="whitegrid", font_scale=1.05)
plt.rcParams.update({
    "figure.dpi":        110,
    "axes.titlesize":    12,
    "axes.titleweight":  "bold",
    "axes.spines.top":   False,
    "axes.spines.right": False,
})
C0 = "#4C9BE8"   # blue  = did not churn
C1 = "#E8654C"   # red   = churned

# StratifiedKFold preserves the ~26%/74% class ratio in every fold.
# 5 folds gives a good bias-variance tradeoff for ~5,600-row training set.
CV           = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
RANDOM_STATE = 42

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.4f}".format)

print(f"xgboost  {xgb.__version__}")
print(f"lightgbm {lgb.__version__}")
print(f"shap     {shap.__version__}")
print("Setup complete.")

---
## 2. Load Features and Target

We load both raw and scaled feature matrices from notebook 03:
- `X_raw` for tree-based models (scale-invariant — they split on thresholds, not distances)
- `X_scaled` for Logistic Regression (requires standardised inputs to converge correctly)

We verify column names against `feature_names.txt` to catch any mismatch between runs of notebook 03. A column-order mismatch would silently corrupt model predictions.

In [ ]:
X_raw    = pd.read_csv(f"{PROCESSED_PATH}X_raw.csv")
X_scaled = pd.read_csv(f"{PROCESSED_PATH}X_scaled.csv")
y        = pd.read_csv(f"{PROCESSED_PATH}y.csv").squeeze().astype(int)

with open(f"{PROCESSED_PATH}feature_names.txt") as fh:
    saved_features = fh.read().splitlines()

assert list(X_raw.columns) == saved_features, \
    f"Column mismatch! Expected {saved_features}, got {list(X_raw.columns)}"

print(f"X_raw    : {X_raw.shape}")
print(f"X_scaled : {X_scaled.shape}")
print(f"y        : {y.shape}  |  churn rate: {y.mean():.1%}")
print(f"Feature columns verified against feature_names.txt ok")
print(f"\nFeature columns ({len(saved_features)}):")
print(X_raw.columns.tolist())

---
## 3. Train / Validation / Test Split

### Why three sets, not two?

A two-way split (train/test) is the most common beginner mistake. Every time you adjust your model after seeing test results — changing a hyperparameter, picking a different model, adjusting a threshold — you are leaking test information into your decisions, even subconsciously. The test set stops being a fair evaluation the moment it influences any decision.

Three-way split:
- **Train (64%)** — models learn from this
- **Validation (16%)** — threshold selection after tuning only
- **Test (20%)** — opened exactly once in Section 10

All CV-based model selection and hyperparameter tuning happens entirely inside the training set.

### Why `stratify=y` in every split?

Without stratification, random chance can allocate most churners to one split. Stratification guarantees every split has the same ~26% churn rate as the full dataset — all evaluations are on representative samples.

In [ ]:
# Step 1: hold out 20% as the untouchable test set
(
    X_raw_trainval,  X_raw_test,
    X_sc_trainval,   X_sc_test,
    y_trainval,      y_test
) = train_test_split(
    X_raw, X_scaled, y,
    test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

# Step 2: split remaining 80% -> 80% train / 20% val = 64% / 16% overall
(
    X_raw_train,  X_raw_val,
    X_sc_train,   X_sc_val,
    y_train,      y_val
) = train_test_split(
    X_raw_trainval, X_sc_trainval, y_trainval,
    test_size=0.20, random_state=RANDOM_STATE, stratify=y_trainval
)

print("Split summary:")
for name, X_, y_ in [
    ("Train",      X_raw_train, y_train),
    ("Validation", X_raw_val,   y_val),
    ("Test",       X_raw_test,  y_test),
]:
    print(f"  {name:<12}  {len(X_):>5,} rows  |  churn rate: {y_.mean():.1%}")

print("\nTest set is locked. Not touched until Section 10.")

---
## 4. Evaluation Metrics — Why Not Accuracy

We define the evaluation framework before training anything. Getting the metric right is more important than getting the model right — the wrong metric will make a useless model look excellent.

**Why NOT accuracy?** A model that always predicts no-churn achieves ~74% accuracy. It identifies exactly zero customers to intervene with. Accuracy is structurally broken for imbalanced classification.

**Why ROC-AUC?** Measures a model's ability to rank churners above non-churners across all possible thresholds. It is threshold-independent and imbalance-robust. A score of 0.5 = random, 1.0 = perfect.

**Why PR-AUC (average precision)?** ROC curves can be misleadingly high on imbalanced data because they include the true negative rate — which is trivially easy to get right when the majority class dominates. PR-AUC focuses only on the positive class and gives a harder, more honest measure of minority-class performance.

**Why recall over precision as the business metric?** Missing a churner who then leaves costs more than incorrectly flagging a loyal customer (who gets an unnecessary retention call but stays). High recall = catch as many churners as possible. This is a business decision, not a statistical one.

In [ ]:
def evaluate(model, X, y_true, threshold=0.5, label=""):
    """
    Standard evaluation for any fitted sklearn classifier with predict_proba().

    Parameters
    ----------
    model     : fitted classifier
    X         : feature matrix
    y_true    : true binary labels
    threshold : probability cutoff for positive prediction
    label     : optional header string

    Returns dict of metric_name -> float
    """
    proba = model.predict_proba(X)[:, 1]
    preds = (proba >= threshold).astype(int)

    metrics = {
        "roc_auc":   round(roc_auc_score(y_true, proba), 4),
        "pr_auc":    round(average_precision_score(y_true, proba), 4),
        "f1":        round(f1_score(y_true, preds, zero_division=0), 4),
        "recall":    round(recall_score(y_true, preds, zero_division=0), 4),
        "precision": round(precision_score(y_true, preds, zero_division=0), 4),
    }

    if label:
        print(f"\n{'─' * 50}")
        print(f"  {label}")
        print(f"{'─' * 50}")
        for k, v in metrics.items():
            print(f"  {k:<12}  {v:.4f}")

    return metrics

print("evaluate() helper defined.")

---
## 5. Baseline Model

**Why train a baseline first?** It defines the floor — the minimum any real model must beat to be considered useful. Without it, a ROC-AUC of 0.72 sounds good but you cannot tell whether it is 0.20 above random or 0.01 above.

**Why `strategy='stratified'`?** `most_frequent` predicts no-churn for everyone (74% accuracy, 0% recall on churners). `stratified` randomly predicts churn proportional to class frequency — a more honest random baseline with non-zero predictions for both classes.

In [ ]:
dummy = DummyClassifier(strategy="stratified", random_state=RANDOM_STATE)
dummy.fit(X_raw_train, y_train)

baseline = evaluate(dummy, X_raw_val, y_val, label="Baseline (stratified random)")
print("\nEvery real model must beat these numbers substantially.")

---
## 6. SMOTE — Prepare Resampled Training Data

SMOTE generates synthetic new churner samples by interpolating between existing churners in feature space.

**Why SMOTE over random over-sampling?** Random duplication gives the model identical rows — it memorises specific churners rather than learning the underlying pattern. SMOTE creates interpolated points between real churners, forcing the model to learn a generalised boundary.

**Why not under-sample the majority class?** With ~7,000 rows, removing non-churners to balance classes cuts training data to ~3,600 rows — discarding real information about what a loyal customer looks like.

**Critical: SMOTE is applied only to training data, never validation or test.** Applying SMOTE to validation or test would add synthetic positive examples that are not representative of real unseen customers and would artificially inflate metrics.

In [ ]:
smote = SMOTE(random_state=RANDOM_STATE, k_neighbors=5)

X_raw_train_sm, y_train_sm = smote.fit_resample(X_raw_train, y_train)
X_sc_train_sm,  _          = smote.fit_resample(X_sc_train,  y_train)

print("Training set class distribution:")
print(f"  Before SMOTE:  {pd.Series(y_train).value_counts().sort_index().to_dict()}")
print(f"  After  SMOTE:  {pd.Series(y_train_sm).value_counts().sort_index().to_dict()}")
print(f"  +{len(y_train_sm) - len(y_train):,} synthetic churner rows added")
print("\nValidation and test sets: unchanged.")

---
## 7. Model Comparison — 5-Fold Cross-Validation

We compare four fundamentally different model families.

| Model | Why include it | Key weakness |
|---|---|---|
| Logistic Regression | Fast, interpretable, reveals if relationships are mostly linear | Assumes linearity |
| Random Forest | Handles non-linearity natively, robust to outliers | Less powerful than boosting |
| XGBoost | Sequentially corrects errors, typically best on tabular data | Many hyperparameters |
| LightGBM | Leaf-wise growth, fastest to train, often best on mid-size tabular data | Can overfit without regularisation |

**Why 5-fold CV instead of a single val score?** A single split has high variance — the exact validation rows might be unusually easy or hard. 5-fold CV averages across five different validation subsets, giving a stable estimate of true generalisation performance.

**Why sort by PR-AUC?** PR-AUC is harder to inflate on imbalanced data. A model genuinely good at finding churners will rank highly on PR-AUC. One that just classifies the majority class correctly will not.

In [ ]:
# scale_pos_weight for XGBoost: equivalent to class_weight='balanced'
# = number of negative samples / number of positive samples
pos_weight = float((y_train == 0).sum()) / float((y_train == 1).sum())
print(f"XGBoost scale_pos_weight: {pos_weight:.2f}")

MODELS = {
    "Logistic Regression": LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        solver="lbfgs",
        random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    "XGBoost": xgb.XGBClassifier(
        scale_pos_weight=pos_weight,
        n_estimators=200,
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        verbosity=0,
        use_label_encoder=False
    ),
    "LightGBM": lgb.LGBMClassifier(
        class_weight="balanced",
        n_estimators=200,
        random_state=RANDOM_STATE,
        verbosity=-1
    )
}
print(f"Models: {list(MODELS.keys())}")

In [ ]:
cv_results = []

for model_name, model in MODELS.items():
    # Tree models: raw features (scale-invariant)
    # Logistic Regression: scaled features (needs standardisation)
    X_cv = X_sc_train if model_name == "Logistic Regression" else X_raw_train

    t0     = time.time()
    scores = cross_validate(
        model, X_cv, y_train,
        cv=CV,
        scoring=["roc_auc", "average_precision", "f1", "recall"],
        return_train_score=False,  # only care about held-out generalisation
        n_jobs=-1
    )
    elapsed = time.time() - t0

    cv_results.append({
        "model":       model_name,
        "roc_auc":     round(scores["test_roc_auc"].mean(), 4),
        "roc_auc_std": round(scores["test_roc_auc"].std(),  4),
        "pr_auc":      round(scores["test_average_precision"].mean(), 4),
        "pr_auc_std":  round(scores["test_average_precision"].std(),  4),
        "f1":          round(scores["test_f1"].mean(), 4),
        "recall":      round(scores["test_recall"].mean(), 4),
        "time_s":      round(elapsed, 1)
    })
    print(f"  {model_name:<22}  ROC-AUC={scores['test_roc_auc'].mean():.4f}  "
          f"PR-AUC={scores['test_average_precision'].mean():.4f}  ({elapsed:.1f}s)")

cv_df = (
    pd.DataFrame(cv_results)
    .sort_values("pr_auc", ascending=False)
    .reset_index(drop=True)
)
print("\nFull CV results (sorted by PR-AUC):")
print(cv_df.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, metric, std_col, title in [
    (axes[0], "roc_auc", "roc_auc_std", "ROC-AUC (5-fold CV)"),
    (axes[1], "pr_auc",  "pr_auc_std",  "PR-AUC  (5-fold CV)")
]:
    sdf    = cv_df.sort_values(metric, ascending=True)
    colors = [C1 if i == len(sdf) - 1 else "#888888" for i in range(len(sdf))]

    bars = ax.barh(
        sdf["model"], sdf[metric],
        xerr=sdf[std_col], color=colors,
        edgecolor="white", linewidth=0.8,
        capsize=4, error_kw={"linewidth": 1.2}
    )
    for bar, val in zip(bars, sdf[metric]):
        ax.text(val + 0.002, bar.get_y() + bar.get_height() / 2,
                f"{val:.4f}", va="center", fontsize=9)

    ax.axvline(0.5, color="grey", linestyle="--", linewidth=0.9, label="Random (0.5)")
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel(metric.upper())
    ax.set_xlim(0, 1)
    ax.legend(fontsize=8)

plt.suptitle("Model comparison - 5-fold cross-validation",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

BEST_MODEL_NAME = cv_df.iloc[0]["model"]
print(f"Best model by PR-AUC: {BEST_MODEL_NAME}")
print("This model proceeds to imbalance comparison and tuning.")

---
## 8. SMOTE vs class_weight Comparison

We compare whether SMOTE or `class_weight='balanced'` produces better CV performance for the best model. We do this comparison only for the best model — running it for all four would create a multiple-comparison problem where we effectively tune the imbalance strategy on the results.

**Why compare these two?** They address imbalance in fundamentally different ways. `class_weight` adjusts the loss function — the model sees the same data but penalises churner misclassification more heavily. SMOTE changes the data — the model sees more churner examples but some are synthetic. Neither is universally superior; the comparison lets the data decide.

In [ ]:
def _fresh_model_no_weight(name):
    """Build the best model WITHOUT class_weight.
    Used for the SMOTE comparison — SMOTE already balances, so adding
    class_weight on top would double-count the imbalance correction."""
    if name == "LightGBM":
        return lgb.LGBMClassifier(
            n_estimators=200, random_state=RANDOM_STATE, verbosity=-1
        )
    elif name == "XGBoost":
        return xgb.XGBClassifier(
            n_estimators=200, eval_metric="logloss",
            random_state=RANDOM_STATE, verbosity=0, use_label_encoder=False
        )
    elif name == "Random Forest":
        return RandomForestClassifier(
            n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1
        )
    else:
        return LogisticRegression(
            max_iter=1000, solver="lbfgs", random_state=RANDOM_STATE
        )

model_cw    = MODELS[BEST_MODEL_NAME]
model_smote = _fresh_model_no_weight(BEST_MODEL_NAME)

X_cw    = X_sc_train    if BEST_MODEL_NAME == "Logistic Regression" else X_raw_train
X_smote = X_sc_train_sm if BEST_MODEL_NAME == "Logistic Regression" else X_raw_train_sm

s_cw = cross_validate(
    model_cw, X_cw, y_train,
    cv=CV, scoring=["roc_auc", "average_precision"], n_jobs=-1
)
s_sm = cross_validate(
    model_smote, X_smote, y_train_sm,
    cv=CV, scoring=["roc_auc", "average_precision"], n_jobs=-1
)

print(f"Imbalance strategy comparison for {BEST_MODEL_NAME}:")
print(f"{'Strategy':<25}  {'ROC-AUC':>10}  {'PR-AUC':>10}")
print("-" * 50)
print(f"{'class_weight=balanced':<25}  "
      f"{s_cw['test_roc_auc'].mean():>10.4f}  "
      f"{s_cw['test_average_precision'].mean():>10.4f}")
print(f"{'SMOTE':<25}  "
      f"{s_sm['test_roc_auc'].mean():>10.4f}  "
      f"{s_sm['test_average_precision'].mean():>10.4f}")

USE_SMOTE = (
    s_sm["test_average_precision"].mean()
    > s_cw["test_average_precision"].mean()
)
CHOSEN_STRATEGY = "SMOTE" if USE_SMOTE else "class_weight=balanced"
print(f"\nChosen strategy: {CHOSEN_STRATEGY}")

X_tune        = X_smote          if USE_SMOTE else X_cw
y_tune        = y_train_sm       if USE_SMOTE else y_train
MODEL_TO_TUNE = model_smote      if USE_SMOTE else model_cw

---
## 9. Hyperparameter Tuning — RandomizedSearchCV

**Why RandomizedSearchCV instead of GridSearchCV?** GridSearchCV tries every combination exhaustively. With 6 parameters each having 5 values, that is 5^6 = 15,625 combinations x 5 folds = 78,125 model fits — many hours on Colab. RandomizedSearchCV samples `n_iter=50` random combinations. Bergstra & Bengio (2012) showed random search finds equally good solutions in a fraction of the time, because most performance gain comes from getting a few key parameters roughly right.

**Why score on `average_precision` (PR-AUC) during tuning?** This is our primary metric. Tuning on ROC-AUC could optimise for a metric that overstates minority-class performance on imbalanced data.

**Why `refit=True`?** After finding best parameters, sklearn automatically refits the model on the full training data — the result is immediately ready for evaluation.

In [ ]:
PARAM_GRIDS = {
    "LightGBM": {
        "n_estimators":      [200, 400, 600, 800],
        "learning_rate":     [0.01, 0.03, 0.05, 0.1, 0.15],
        "max_depth":         [3, 4, 5, 6, 7, -1],
        "num_leaves":        [20, 31, 50, 70, 100],
        "min_child_samples": [10, 20, 30, 50],
        "feature_fraction":  [0.6, 0.7, 0.8, 0.9, 1.0],
        "subsample":         [0.6, 0.7, 0.8, 0.9, 1.0],
        "reg_alpha":         [0, 0.01, 0.1, 1.0],
        "reg_lambda":        [0, 0.1, 1.0, 5.0]
    },
    "XGBoost": {
        "n_estimators":     [200, 400, 600, 800],
        "learning_rate":    [0.01, 0.03, 0.05, 0.1, 0.15],
        "max_depth":        [3, 4, 5, 6, 7],
        "min_child_weight": [1, 3, 5, 10],
        "subsample":        [0.6, 0.7, 0.8, 0.9, 1.0],
        "colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0],
        "gamma":            [0, 0.1, 0.5, 1.0],
        "reg_alpha":        [0, 0.1, 1.0],
        "reg_lambda":       [1.0, 2.0, 5.0]
    },
    "Random Forest": {
        "n_estimators":      [200, 400, 600],
        "max_depth":         [None, 8, 12, 16, 20],
        "min_samples_split": [2, 5, 10, 20],
        "min_samples_leaf":  [1, 2, 4, 8],
        "max_features":      ["sqrt", "log2", 0.5, 0.7]
    },
    "Logistic Regression": {
        "C":       [0.001, 0.01, 0.1, 1, 10, 100],
        "penalty": ["l1", "l2"],
        "solver":  ["liblinear", "saga"]
    }
}

param_grid = PARAM_GRIDS.get(BEST_MODEL_NAME, {})
print(f"Tuning: {BEST_MODEL_NAME}")
print(f"Strategy: {CHOSEN_STRATEGY}")
print(f"50 iterations x 5 folds = 250 fits")

In [ ]:
print("Running RandomizedSearchCV... (5-15 minutes on Colab)")

t0 = time.time()
search = RandomizedSearchCV(
    MODEL_TO_TUNE,
    param_distributions=param_grid,
    n_iter=50,
    scoring="average_precision",
    cv=CV,
    refit=True,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=0
)
search.fit(X_tune, y_tune)
elapsed = time.time() - t0

print(f"\nTuning complete in {elapsed:.0f}s")
print(f"Best PR-AUC (CV): {search.best_score_:.4f}")
print("\nBest parameters:")
for k, v in sorted(search.best_params_.items()):
    print(f"  {k:<25}  {v}")

BEST_MODEL = search.best_estimator_

### 9b. Validation set check after tuning

We evaluate on validation before opening test. This confirms tuning helped and gives us a final pre-test estimate. If validation numbers are substantially worse than the CV score, the parameter search overfit to the cross-validation folds — a sign to reduce `n_iter` or expand the parameter ranges.

In [ ]:
X_val_eval = X_sc_val if BEST_MODEL_NAME == "Logistic Regression" else X_raw_val

tuned_val = evaluate(
    BEST_MODEL, X_val_eval, y_val,
    label=f"{BEST_MODEL_NAME} (tuned) - Validation set"
)

untuned_pr = cv_df[cv_df["model"] == BEST_MODEL_NAME]["pr_auc"].values[0]
print(f"\nPR-AUC: before tuning (CV mean) = {untuned_pr:.4f}  "
      f"-> after tuning (val) = {tuned_val['pr_auc']:.4f}  "
      f"({tuned_val['pr_auc'] - untuned_pr:+.4f})")

---
## 10. Decision Threshold Optimisation

**Why optimise the threshold?** Sklearn's default threshold of 0.5 was not chosen to match any business objective. For a churn model prioritising recall, the optimal threshold is almost always below 0.5.

**Why use the validation set?** The threshold is a model decision being fitted to data. Using the test set would leak test information. Validation is the correct choice.

**What it controls:** lower threshold = higher recall (more churners caught) but lower precision (more false alarms). We find the F1-optimal threshold as the default operating point and show the full tradeoff curve so a business analyst can choose a different point if they want to prioritise recall even further.

In [ ]:
proba_val = BEST_MODEL.predict_proba(X_val_eval)[:, 1]

precisions, recalls, thresholds = precision_recall_curve(y_val, proba_val)

# F1 at each threshold
f1_scores = np.where(
    (precisions[:-1] + recalls[:-1]) > 0,
    2 * precisions[:-1] * recalls[:-1] / (precisions[:-1] + recalls[:-1]),
    0
)
best_idx       = int(np.argmax(f1_scores))
BEST_THRESHOLD = float(thresholds[best_idx])

print("Threshold comparison (validation set):")
_ = evaluate(BEST_MODEL, X_val_eval, y_val,
             threshold=0.50, label="Default threshold = 0.50")
_ = evaluate(BEST_MODEL, X_val_eval, y_val,
             threshold=BEST_THRESHOLD,
             label=f"Optimal threshold = {BEST_THRESHOLD:.3f} (F1-optimal)")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(thresholds, precisions[:-1], color=C0, linewidth=2, label="Precision")
axes[0].plot(thresholds, recalls[:-1],    color=C1, linewidth=2, label="Recall")
axes[0].plot(thresholds, f1_scores,       color="#6BBF59", linewidth=2,
             linestyle="--", label="F1")
axes[0].axvline(BEST_THRESHOLD, color="black", linestyle=":", linewidth=1.5,
                label=f"Optimal ({BEST_THRESHOLD:.3f})")
axes[0].set_xlabel("Threshold")
axes[0].set_ylabel("Score")
axes[0].set_title("Precision / Recall / F1 vs threshold", fontweight="bold")
axes[0].legend(fontsize=9)
axes[0].set_xlim(0, 1)

PrecisionRecallDisplay.from_predictions(
    y_val, proba_val, ax=axes[1], name=BEST_MODEL_NAME, color=C1
)
axes[1].axhline(y_val.mean(), color="grey", linestyle="--",
                linewidth=1, label=f"Random ({y_val.mean():.2f})")
axes[1].set_title("Precision-Recall curve (validation)", fontweight="bold")
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

print(f"\nFinal decision threshold: {BEST_THRESHOLD:.3f}")
print("Applied to all test set and business scoring outputs.")

---
## 11. Final Test Set Evaluation

### The test set is opened exactly once — right here.

Every decision above — model selection, SMOTE comparison, tuning, threshold selection — was made without looking at the test set. The numbers below are the true generalisation performance of the final model on completely unseen data. If they are substantially worse than validation numbers, the model overfit to the validation set. If similar, it generalises well.

In [ ]:
X_test_eval = X_sc_test if BEST_MODEL_NAME == "Logistic Regression" else X_raw_test

print("=" * 55)
print("FINAL TEST SET EVALUATION")
print(f"Model:     {BEST_MODEL_NAME}")
print(f"Strategy:  {CHOSEN_STRATEGY}")
print(f"Threshold: {BEST_THRESHOLD:.3f}")
print("=" * 55)

test_metrics = evaluate(
    BEST_MODEL, X_test_eval, y_test,
    threshold=BEST_THRESHOLD,
    label=f"{BEST_MODEL_NAME} - TEST SET (final, one-time)"
)

proba_test = BEST_MODEL.predict_proba(X_test_eval)[:, 1]
preds_test = (proba_test >= BEST_THRESHOLD).astype(int)

print("\nClassification report:")
print(classification_report(
    y_test, preds_test,
    target_names=["Did not churn", "Churned"]
))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

RocCurveDisplay.from_predictions(
    y_test, proba_test, ax=axes[0],
    name=BEST_MODEL_NAME, color=C1
)
axes[0].plot([0, 1], [0, 1], "--", color="grey", linewidth=1, label="Random")
axes[0].set_title("ROC Curve - Test Set", fontweight="bold")
axes[0].legend(fontsize=9)

PrecisionRecallDisplay.from_predictions(
    y_test, proba_test, ax=axes[1],
    name=BEST_MODEL_NAME, color=C1
)
axes[1].axhline(y_test.mean(), color="grey", linestyle="--",
                linewidth=1, label="Random baseline")
axes[1].set_title("Precision-Recall - Test Set", fontweight="bold")
axes[1].legend(fontsize=9)

cm = confusion_matrix(y_test, preds_test)
ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["No churn", "Churned"]
).plot(ax=axes[2], cmap="Blues", colorbar=False)
axes[2].set_title("Confusion Matrix - Test Set", fontweight="bold")

plt.suptitle(f"Final test evaluation - {BEST_MODEL_NAME}",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print("Confusion matrix:")
print(f"  True Positives  (churners flagged):        {tp:>5,}")
print(f"  True Negatives  (loyal customers kept):    {tn:>5,}")
print(f"  False Positives (loyal flagged as risk):   {fp:>5,}")
print(f"  False Negatives (churners missed):         {fn:>5,}")
print(f"\n  Catch rate (recall): {tp/(tp+fn):.1%} of all churners identified")
print(f"  False alarm rate:    {fp/(fp+tn):.1%} of loyal customers incorrectly flagged")

---
## 12. Model Interpretation — SHAP Values

**Why SHAP and not just feature importance?**

Built-in feature importance (e.g. `feature_importances_`) has two critical flaws:
1. It overstates high-cardinality features
2. It gives one global score per feature — it cannot show whether a feature increases or decreases churn probability

SHAP (SHapley Additive exPlanations) solves both:
1. Grounded in cooperative game theory — each feature gets its fair marginal contribution
2. Shows directionality: positive SHAP pushes toward churn, negative pushes away
3. Provides both global rankings and individual-customer explanations

**Why `TreeExplainer`?** For tree-based models it computes exact SHAP values in polynomial time by exploiting the tree structure — orders of magnitude faster than the model-agnostic `KernelExplainer`. For Logistic Regression we use `LinearExplainer`.

In [ ]:
print("Computing SHAP values on test set...")

if BEST_MODEL_NAME in ["LightGBM", "XGBoost", "Random Forest"]:
    explainer = shap.TreeExplainer(BEST_MODEL)
else:
    # LinearExplainer: exact and fast for linear models
    explainer = shap.LinearExplainer(BEST_MODEL, X_sc_train)

shap_values = explainer.shap_values(X_test_eval)

# LightGBM returns [class_0_shap, class_1_shap] - we want class 1 (churn)
if isinstance(shap_values, list):
    sv = shap_values[1]
else:
    sv = shap_values

feat_names = X_raw.columns.tolist()
print(f"SHAP values shape: {sv.shape}  (test rows x features)")
print("Done.")

### 12a. Global feature importance — beeswarm plot

Each dot is one test-set customer. The x-axis is the SHAP value (positive = pushes toward churn). Colour is the feature's actual value (red = high, blue = low). Features are sorted by mean absolute SHAP value.

How to read it: if a feature has red dots on the right and blue dots on the left, high values of that feature increase churn probability. If the colours are reversed, high values reduce churn risk.

In [ ]:
shap.summary_plot(
    sv, X_test_eval,
    feature_names=feat_names,
    max_display=20,
    show=True
)
plt.title("SHAP beeswarm - global feature importance", fontweight="bold")
plt.tight_layout()
plt.show()

### 12b. Mean absolute SHAP bar chart

One number per feature: the average absolute SHAP value across all test customers. This is the clean executive-facing summary — how much, on average, does each feature move a customer's churn probability?

In [ ]:
mean_abs_shap = (
    pd.Series(np.abs(sv).mean(axis=0), index=feat_names)
    .sort_values(ascending=False)
)

top_n    = 15
top_shap = mean_abs_shap.head(top_n)
colors   = [C1 if i < 5 else "#888888" for i in range(top_n)]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(
    top_shap.index[::-1], top_shap.values[::-1],
    color=colors[::-1], edgecolor="white", linewidth=0.8
)
ax.set_xlabel("Mean |SHAP value| (avg impact on churn probability)")
ax.set_title(f"Top {top_n} features - SHAP global importance", fontweight="bold")
for i, (val, _) in enumerate(zip(top_shap.values[:5], top_shap.index[:5])):
    ax.text(val + 0.0005, top_n - 1 - i, f"{val:.4f}", va="center", fontsize=8.5)
plt.tight_layout()
plt.show()

### 12c. SHAP dependence plots — top 4 features

A dependence plot shows the relationship between a feature's actual value (x-axis) and its SHAP contribution (y-axis). The colour encodes a second feature that SHAP identified as having the strongest interaction effect. This reveals non-linear relationships and interactions that a correlation coefficient would miss completely.

In [ ]:
top4       = mean_abs_shap.head(4).index.tolist()
fig, axes  = plt.subplots(2, 2, figsize=(14, 10))
axes       = axes.flatten()

for i, feat in enumerate(top4):
    shap.dependence_plot(
        feat, sv, X_test_eval,
        feature_names=feat_names,
        ax=axes[i], show=False
    )
    axes[i].set_title(f"SHAP dependence: {feat}", fontweight="bold", fontsize=10)

plt.suptitle("SHAP dependence plots - top 4 features",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

### 12d. Individual customer explanations — waterfall plots

SHAP waterfall plots explain one specific prediction. This is what a customer success team needs — not just 'this customer is at risk' but exactly *why*: is it the contract type? The billing gap? The low login frequency? Each bar shows how much one feature pushed this specific customer's probability above or below the base rate.

In [ ]:
# Top 3 highest predicted churn probability customers in the test set
top_risk_idx = np.argsort(proba_test)[-3:][::-1]

base_val = explainer.expected_value
if isinstance(base_val, (list, np.ndarray)):
    base_val = float(base_val[1])  # class 1 for LightGBM
else:
    base_val = float(base_val)

for rank, idx in enumerate(top_risk_idx, start=1):
    exp = shap.Explanation(
        values        = sv[idx],
        base_values   = base_val,
        data          = X_test_eval.iloc[idx].values,
        feature_names = feat_names
    )
    print(f"\nAt-risk customer #{rank}")
    print(f"  Predicted churn probability: {proba_test[idx]:.1%}")
    print(f"  Actual churn:                {'YES' if y_test.iloc[idx] == 1 else 'no'}")
    shap.waterfall_plot(exp, max_display=12, show=True)
    plt.tight_layout()
    plt.show()

---
## 13. At-Risk Customer Scoring — Business Deliverable

This is what the business actually uses. We produce a prioritised intervention list for every test-set customer:
- Churn probability score
- Risk tier (High / Medium / Low) for easy prioritisation
- Top 3 personalised churn drivers from SHAP

**Why risk tiers?** Raw probabilities are hard for non-technical teams to act on. A tier system gives the customer success team a clear queue.

**Why SHAP-based drivers?** A retention team cannot personalise a conversation based only on 'the model flagged this customer'. They need to know what to address. Is it contract type? Late payments? No engagement? SHAP tells them exactly.

In [ ]:
scoring = pd.DataFrame({
    "churn_probability": proba_test,
    "predicted_churn":   preds_test,
    "actual_churn":      y_test.values
})

scoring["risk_tier"] = pd.cut(
    scoring["churn_probability"],
    bins=[0, 0.35, 0.60, 1.0],
    labels=["Low", "Medium", "High"]
)

def top_drivers(shap_row, feature_names, n=3):
    """Features with the largest positive SHAP values for this customer."""
    top_idx = np.argsort(shap_row)[-n:][::-1]
    return ", ".join([feature_names[i] for i in top_idx])

scoring["top_churn_drivers"] = [
    top_drivers(sv[i], feat_names)
    for i in range(len(proba_test))
]

scoring = scoring.sort_values("churn_probability", ascending=False).reset_index(drop=True)

print(f"Scoring table: {scoring.shape}")
print("\nRisk tier distribution:")
print(scoring["risk_tier"].value_counts().sort_index().to_string())
print("\nTop 10 highest-risk customers:")
print(scoring.head(10).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
tier_colors = ["#4C9BE8", "#F5A623", "#E8654C"]

# Actual churn rate by risk tier
tier_stats = (
    scoring.groupby("risk_tier", observed=True)["actual_churn"]
    .agg(churn_rate="mean", n="count")
    .reset_index()
)
bars = axes[0].bar(
    tier_stats["risk_tier"].astype(str),
    tier_stats["churn_rate"],
    color=tier_colors, edgecolor="white", linewidth=1.5
)
for bar, (_, row) in zip(bars, tier_stats.iterrows()):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.01,
        f"{row['churn_rate']:.0%}\nn={row['n']:,}",
        ha="center", va="bottom", fontsize=10
    )
axes[0].axhline(y_test.mean(), color="grey", linestyle="--",
                linewidth=1, label=f"Overall {y_test.mean():.0%}")
axes[0].set_title("Actual churn rate by risk tier", fontweight="bold")
axes[0].set_xlabel("Risk tier")
axes[0].set_ylabel("Churn rate")
axes[0].yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
axes[0].legend(fontsize=9)

# Probability distribution by tier
for tier, color in zip(["Low", "Medium", "High"], tier_colors):
    subset = scoring[scoring["risk_tier"] == tier]["churn_probability"]
    axes[1].hist(subset, bins=25, color=color, alpha=0.6,
                 label=tier, edgecolor="white")
axes[1].axvline(BEST_THRESHOLD, color="black", linestyle=":", linewidth=1.5,
                label=f"Threshold ({BEST_THRESHOLD:.2f})")
axes[1].set_title("Churn probability distribution by tier", fontweight="bold")
axes[1].set_xlabel("Predicted churn probability")
axes[1].set_ylabel("Customers")
axes[1].legend(fontsize=9)

plt.suptitle("At-risk customer scoring - business view",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

---
## 14. Save All Artefacts

We save everything needed to reproduce results and deploy the model. The scaler is refit here on training data only — the scaler saved in notebook 03 was technically fit on the full dataset, which is a minor leakage. This version is genuinely production-safe.

In [ ]:
# Refit scaler on training data only (correct, leak-free)
scaler_final = StandardScaler()
cols_to_scale = [
    c for c in X_raw_train.columns
    if X_raw_train[c].dtype in ["float64", "float32"]
    or (X_raw_train[c].dtype in ["int64", "int32"]
        and X_raw_train[c].nunique() > 10)
]
scaler_final.fit(X_raw_train[cols_to_scale])

# Compile validation + test metrics into one table
metrics_df = pd.DataFrame([
    {"split": "validation", **tuned_val},
    {"split": "test",       **test_metrics}
])

artefacts = {
    "model":   f"{MODELS_PATH}best_model.pkl",
    "scaler":  f"{MODELS_PATH}scaler_final.pkl",
    "scoring": f"{PROCESSED_PATH}scoring_table.csv",
    "metrics": f"{PROCESSED_PATH}model_metrics.csv",
    "thresh":  f"{MODELS_PATH}threshold.txt",
    "meta":    f"{MODELS_PATH}model_info.txt",
}

joblib.dump(BEST_MODEL,   artefacts["model"])
joblib.dump(scaler_final, artefacts["scaler"])
scoring.to_csv(           artefacts["scoring"], index=False)
metrics_df.to_csv(        artefacts["metrics"], index=False)

with open(artefacts["thresh"], "w") as fh:
    fh.write(str(BEST_THRESHOLD))

with open(artefacts["meta"], "w") as fh:
    lines = [
        f"model_name: {BEST_MODEL_NAME}",
        f"imbalance_strategy: {CHOSEN_STRATEGY}",
        f"threshold: {BEST_THRESHOLD:.4f}",
        f"test_roc_auc: {test_metrics['roc_auc']:.4f}",
        f"test_pr_auc: {test_metrics['pr_auc']:.4f}",
        f"test_recall: {test_metrics['recall']:.4f}",
        f"test_f1: {test_metrics['f1']:.4f}",
        f"best_params: {search.best_params_}",
    ]
    fh.write("\n".join(lines))

print("Saved artefacts:")
for name, path in artefacts.items():
    size_kb = os.path.getsize(path) / 1024
    print(f"  {os.path.basename(path):<30}  {size_kb:>8.1f} KB")

---
## 15. Phase 4 Summary

In [ ]:
print("PHASE 4 COMPLETE - MODELLING SUMMARY")
print("=" * 55)
print()
print("Pipeline:")
print("  Split:    64% train / 16% validation / 20% test (stratified)")
print("  Metrics:  ROC-AUC + PR-AUC (primary) | F1, Recall (secondary)")
print("  Baseline: DummyClassifier(strategy=stratified)")
print("  4 models compared via 5-fold stratified CV")
print(f"  Best model:          {BEST_MODEL_NAME}")
print(f"  Imbalance strategy:  {CHOSEN_STRATEGY}")
print(f"  Tuning:              RandomizedSearchCV (50 iters x 5 folds, PR-AUC)")
print(f"  Threshold:           {BEST_THRESHOLD:.3f} (F1-optimal on validation set)")
print()
print("Final test set results:")
for k, v in test_metrics.items():
    print(f"  {k:<12}  {v:.4f}")
print()
print("Business output:")
high_n = (scoring["risk_tier"] == "High").sum()
med_n  = (scoring["risk_tier"] == "Medium").sum()
low_n  = (scoring["risk_tier"] == "Low").sum()
print(f"  High risk:   {high_n:,}  customers")
print(f"  Medium risk: {med_n:,}  customers")
print(f"  Low risk:    {low_n:,}  customers")
print(f"  Each flagged with top 3 personalised SHAP-based churn drivers")
print()
print("Artefacts saved:")
print("  models/best_model.pkl      trained model")
print("  models/scaler_final.pkl    scaler fit on training data only")
print("  models/threshold.txt       optimal decision threshold")
print("  models/model_info.txt      full metadata for reproducibility")
print("  processed/scoring_table.csv  at-risk list with SHAP drivers")
print("  processed/model_metrics.csv  validation + test metrics")
print()
print("Project complete.")